This question should be answered using the Weekly data set, which
is part of the ISLP package. This data is similar in nature to the
Smarket data from this chapter’s lab, except that it contains 1, 089
weekly returns for 21 years, from the beginning of 1990 to the end of
2010.

(e) Repeat (d) using LDA.

In [20]:
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

Weekly = pd.read_csv('Weekly.csv').copy()

# transfer: Direction -> 1:Up, 0:Down
y = Weekly['Direction'].map({'Down': 0, 'Up': 1}).astype(int)
labels = [0, 1]  # 0=Down, 1=Up

# time-split
train_idx = Weekly['Year'] <= 2008
test_idx  = Weekly['Year'] >= 2009

# lag2
X_train_lag2 = Weekly.loc[train_idx, ['Lag2']].to_numpy()
X_test_lag2  = Weekly.loc[test_idx,  ['Lag2']].to_numpy()
y_train = y.loc[train_idx].to_numpy()
y_test  = y.loc[test_idx].to_numpy()

def report_result(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(f"\n=== {name} ===")
    print("Confusion matrix:")
    print(f"               Actual:Down  Actual:Up")
    print(f"Pred:Down            {cm[0,0]}          {cm[1,0]}")
    print(f"Pred:Up             {cm[0,1]}         {cm[1,1]}")
    print(f"Overall accuracy = {acc:.4f}")
    return acc, cm

In [24]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_lag2, y_train)
y_hat_lda = lda.predict(X_test_lag2)
acc_lda, cm_lda = report_result("LDA (Lag2 only)", y_test, y_hat_lda)


=== LDA (Lag2 only) ===
Confusion matrix:
               Actual:Down  Actual:Up
Pred:Down            9          5
Pred:Up             34         56
Overall accuracy = 0.6250


(f) Repeat (d) using QDA.

In [23]:
qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train_lag2, y_train)
y_hat_qda = qda.predict(X_test_lag2)
acc_qda, cm_qda = report_result("QDA (Lag2 only)", y_test, y_hat_qda)


=== QDA (Lag2 only) ===
Confusion matrix:
               Actual:Down  Actual:Up
Pred:Down            0          0
Pred:Up             43         61
Overall accuracy = 0.5865


(g) Repeat (d) using KNN with K = 1.

In [25]:
knn1 = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=1))
])
knn1.fit(X_train_lag2, y_train)
y_hat_knn1 = knn1.predict(X_test_lag2)
acc_knn1, cm_knn1 = report_result("KNN K=1 (Lag2 only, scaled)", y_test, y_hat_knn1)



=== KNN K=1 (Lag2 only, scaled) ===
Confusion matrix:
               Actual:Down  Actual:Up
Pred:Down            22          32
Pred:Up             21         29
Overall accuracy = 0.4904


(h) Repeat (d) using naive Bayes.

In [26]:
nb = GaussianNB()
nb.fit(X_train_lag2, y_train)
y_hat_nb = nb.predict(X_test_lag2)
acc_nb, cm_nb = report_result("Naive Bayes (Lag2 only)", y_test, y_hat_nb)



=== Naive Bayes (Lag2 only) ===
Confusion matrix:
               Actual:Down  Actual:Up
Pred:Down            0          0
Pred:Up             43         61
Overall accuracy = 0.5865


(i) Which of these methods appears to provide the best results on
this data?

In [28]:
acc_table = pd.DataFrame({
    "model": ["LDA", "QDA", "KNN(k=1)", "NaiveBayes"],
    "accuracy": [acc_lda, acc_qda, acc_knn1, acc_nb]
}).sort_values("accuracy", ascending=False)

print("\n=== Test Accuracy Ranking (Lag2 only) ===")
print(acc_table.to_string(index=False))



=== Test Accuracy Ranking (Lag2 only) ===
     model  accuracy
       LDA  0.625000
       QDA  0.586538
NaiveBayes  0.586538
  KNN(k=1)  0.490385


(j) Experiment with different combinations of predictors, includ-
ing possible transformations and interactions, for each of the
methods. Report the variables, method, and associated confu-
sion matrix that appears to provide the best results on the held
out data. Note that you should also experiment with values for
K in the KNN classifier.

In [36]:
predictor_sets = {
    "Lag2": ["Lag2"],
    "Lags(1-5)+Vol": ["Lag1","Lag2","Lag3","Lag4","Lag5","Volume"],
    "Lags(1-5)": ["Lag1","Lag2","Lag3","Lag4","Lag5"],
}

poly_choices = [False]
knn_ks = [1,3,5,7,9]

def build_X(df, cols, poly=False, fit_to=None):
    X = df[cols].to_numpy()
    if poly:
        pf = PolynomialFeatures(degree=2, include_bias=False)
        if fit_to is None:
            X = pf.fit_transform(X)
            return X, pf
        pf.fit(fit_to[cols].to_numpy())
        X = pf.transform(X)
        return X, pf
    return X, None

records = []

for set_name, cols in predictor_sets.items():
    for poly in poly_choices:
        X_train_all, pf = build_X(Weekly.loc[train_idx, cols], cols, poly=poly, fit_to=Weekly.loc[train_idx])
        X_test_all, _  = build_X(Weekly.loc[test_idx,  cols], cols, poly=poly, fit_to=Weekly.loc[train_idx])

        # LDA
        m = LinearDiscriminantAnalysis().fit(X_train_all, y_train)
        pred = m.predict(X_test_all)
        records.append(("LDA", set_name, poly, None,
                        accuracy_score(y_test, pred),
                        confusion_matrix(y_test, pred, labels=labels)))

        # QDA
        m = QuadraticDiscriminantAnalysis().fit(X_train_all, y_train)
        pred = m.predict(X_test_all)
        records.append(("QDA", set_name, poly, None,
                        accuracy_score(y_test, pred),
                        confusion_matrix(y_test, pred, labels=labels)))

        # Naive Bayes
        m = GaussianNB().fit(X_train_all, y_train)
        pred = m.predict(X_test_all)
        records.append(("NaiveBayes", set_name, poly, None,
                        accuracy_score(y_test, pred),
                        confusion_matrix(y_test, pred, labels=labels)))

        # KNN
        for k in knn_ks:
            m = Pipeline([("scaler", StandardScaler()),
                          ("knn", KNeighborsClassifier(n_neighbors=k))]).fit(X_train_all, y_train)
            pred = m.predict(X_test_all)
            records.append((f"KNN(k={k})", set_name, poly, k,
                            accuracy_score(y_test, pred),
                            confusion_matrix(y_test, pred, labels=labels)))

# summary
res = pd.DataFrame(records, columns=["model","predictors","poly(d2/interactions)","K","accuracy","confusion_matrix"])
res_sorted = res.sort_values("accuracy", ascending=False)

print("\n=== (j) Top candidates on 2009–2010 test set ===")
print(res_sorted.head(12)[["model","predictors","poly(d2/interactions)","K","accuracy"]].to_string(index=False))

best = res_sorted.iloc[0]
print("\nBest setting detail:")
print(best[["model","predictors","poly(d2/interactions)","K","accuracy"]])
print("Confusion matrix:")
cm = best["confusion_matrix"]
print(f"               Actual:Down  Actual:Up")
print(f"Pred:Down           {cm[0,0]}         {cm[0,1]}")
print(f"Pred:Up             {cm[1,0]}         {cm[1,1]}")


=== (j) Top candidates on 2009–2010 test set ===
     model    predictors  poly(d2/interactions)   K  accuracy
       LDA          Lag2                  False NaN  0.625000
       QDA          Lag2                  False NaN  0.586538
NaiveBayes          Lag2                  False NaN  0.586538
  KNN(k=3)          Lag2                  False 3.0  0.557692
  KNN(k=9)          Lag2                  False 9.0  0.557692
  KNN(k=7)          Lag2                  False 7.0  0.557692
  KNN(k=7)     Lags(1-5)                  False 7.0  0.557692
  KNN(k=9) Lags(1-5)+Vol                  False 9.0  0.557692
  KNN(k=3)     Lags(1-5)                  False 3.0  0.557692
       LDA     Lags(1-5)                  False NaN  0.548077
  KNN(k=9)     Lags(1-5)                  False 9.0  0.548077
  KNN(k=7) Lags(1-5)+Vol                  False 7.0  0.548077

Best setting detail:
model                      LDA
predictors                Lag2
poly(d2/interactions)    False
K                          Na